# 1 · Data, audits and feature caches

Downloads Tsinghua Dogs and Stanford Dogs from their owners, verifies them, builds the manifests and the
within-dataset audits, caches the frozen-backbone features every later step reads, and runs the cross-dataset
overlap audit that fixes the provenance subsets of the Tsinghua test split. Nothing here trains a model or
evaluates anything on a test split.

* **Licences.** Tsinghua Dogs: academic use, citation requested. Stanford Dogs: ImageNet terms
  (non-commercial research and education, no redistribution). Nothing here redistributes images.
* **Stanford Dogs source.** If vision.stanford.edu does not answer, `scripts/download_data.sh` falls back
  to Internet Archive snapshots of the official URLs. It accepts a file only if its SHA-256 equals the
  checksum TensorFlow Datasets records for the official release.
* **Splits.** The validation split is 10 % of the official training images (stratified, seed 0).
  Training images that duplicate a test image are excluded. The test split is the official one (for
  Tsinghua Dogs, the official validation list).

## Setup (run at the start of every session)

Runtime → Change runtime type → **T4 GPU** (any CUDA GPU works; the study uses a free-tier T4, 15 GB).
Mounting Google Drive is optional but strongly advised: `scripts/colab_sync.sh` mirrors everything under
`/content/work` (manifests, audits, runs, freezes, results, logs, queue state) to Drive, so a recycled runtime
loses only the feature caches. Notebook 1 rebuilds those from the images. `push()` (below) first copies every
token PCA (`pca.npz`) into `/content/work/pca_backup`, and this cell puts it back, so rebuilt caches use the
basis the heads were trained in (`extract_features.py` reuses an existing `pca.npz` instead of fitting a new one).

**Long jobs run detached.** A cell `!nohup cmd &` stays busy, and keeps the kernel busy, until the job ends,
because the job holds the cell's terminal. Every long job here is started as
`!setsid nohup bash -c '…' > /content/work/logs/<name>.log 2>&1 < /dev/null &`: the cell returns at once, and a
watch cell reads the log. The same commands can be typed into Colab's Terminal panel (without the `!`) after
exporting the variables `scripts/colab_env.py` sets, since a terminal does not have the kernel's environment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # optional: skip this cell to keep everything on the runtime's disk

In [ ]:
# Where the code comes from: a git URL, or a zip of this repository (top folder DogBreed-PhD-Research)
REPO_URL = ""                                                   # e.g. the URL of your fork
REPO_ZIP = "/content/drive/MyDrive/DogBreed-PhD-Research.zip"
import os, subprocess
D = "/content/DogBreed-PhD-Research"                            # the queue files cd here: keep this path
if not os.path.isdir(D):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, D], check=True)
    elif os.path.isfile(REPO_ZIP):
        subprocess.run(["unzip", "-q", REPO_ZIP, "-d", "/content"], check=True)
    else:
        raise SystemExit("set REPO_URL, or put the repository zip at REPO_ZIP")
os.chdir(D)
exec(open("scripts/colab_env.py").read())                     # paths: caches on local disk, the rest on Drive
!pip install -q --no-deps -e . && python -m pytest -q tests 2>&1 | tail -2
!bash scripts/colab_sync.sh pull                                # restore earlier sessions' work, if any
!rsync -a /content/work/pca_backup/ /content/features/ 2>/dev/null && find /content/features -name pca.npz
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


def push():
    """Copy every token PCA of the caches into /content/work/pca_backup, then mirror /content/work to Drive."""
    !rsync -am --include='*/' --include='pca.npz' --include='pca.json' --exclude='*' /content/features/ /content/work/pca_backup/
    !bash scripts/colab_sync.sh push

## 1. Download, verify, manifests, within-dataset audits

`scripts/colab_data.sh all` copies the archives already saved on Drive, downloads the rest
(`scripts/download_data.sh`: resumed downloads, SHA-256 records), saves new archives to Drive, unpacks them and
runs `scripts/prepare_data.py` for each dataset. That writes the manifest (one row per image: label, official
split, our split, decode status, hashes) and the audit: exact and near duplicates (SHA-1, pHash, dHash),
training images that duplicate a test image (excluded from training), near pairs with different labels.
The log ends with `DATA DONE (all)`.

In [ ]:
!setsid nohup bash -c 'bash scripts/colab_data.sh all' > /content/work/logs/data.log 2>&1 < /dev/null &

In [ ]:
!tail -n 4 /content/work/logs/data.log

When the log ends with `DATA DONE (all)`:

In [ ]:
!head -n 60 $DOGBREED_RESULTS/audit/tsinghua.json
push()

## 2. Feature caches (DINOv2 ViT-B/14 with registers, 224 px, frozen)

`scripts/extract_all.sh <dataset> <backbone> <images> [token_pca] [img_size]`:
* Global descriptor: g = [CLS ; mean patch], 1536-d.
* Tokens: the 256 patch tokens, projected on a 256-component PCA. The PCA is fitted on 4,000 *training*
  images only.
* Views:
  * orig: train, val and test;
  * hflip: train only;
  * five automatic corruptions of the test split (blur, noise, jpeg, lowres, occlude);
  * five annotation-assisted test views (head_occlude, bg_gray, bg_blur, crop_body, crop_head). These use the
    dataset's boxes, so they are robustness probes, never automatic inference.

Every cache resumes after an interruption: start the cell again. The log ends with
`EXTRACT DONE tsinghua dinov2_vitb14_reg`. Training (notebook 2) reads the orig and hflip caches; the test views
are read only by the locked evaluation (notebook 3).

In [ ]:
!setsid nohup bash -c 'bash scripts/extract_all.sh tsinghua dinov2_vitb14_reg /content/data/raw/tsinghua_dogs/low-resolution.zip 256 224' > /content/work/logs/extract_tsinghua.log 2>&1 < /dev/null &

In [ ]:
!tail -n 3 /content/work/logs/extract_tsinghua.log; nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

When it is done:

In [ ]:
push()

## 3. Stanford Dogs caches and the cross-dataset audit

Run after the Tsinghua caches (the transfer cache reuses the Tsinghua PCA). `scripts/pipeline_stanford.sh`:
1. the Stanford caches in Stanford's own PCA basis (orig, hflip, the automatic test views, and the
   annotation-assisted body-box views bg_gray, bg_blur, crop_body: Stanford Dogs has no head boxes);
2. the Stanford test split in the Tsinghua PCA basis (feature tag `pca256_tsinghua`), for the transfer study;
3. `scripts/cross_overlap.py`: the Stanford Dogs photographs inside Tsinghua Dogs, from two signals (pHash
   distance and the cosine of the frozen descriptors g), with a threshold-sensitivity table and montages per
   similarity band for a visual check. It also writes the de-duplicated Stanford test list for the transfer study.

The log ends with `STANFORD PIPELINE DONE`. Queue B (notebook 2) waits for this script to end before it trains
`stanford_main`, so notebook 2 can be started as soon as the Tsinghua caches are done.

In [ ]:
!setsid nohup bash -c 'bash scripts/pipeline_stanford.sh' > /content/work/logs/pipeline_stanford.log 2>&1 < /dev/null &

In [ ]:
!tail -n 3 /content/work/logs/pipeline_stanford.log

If `cross_overlap.py` runs out of GPU memory (training jobs share the GPU), run the audit alone on the CPU: the
same computation, only slower.

In [ ]:
!setsid nohup bash -c 'python -I scripts/cross_overlap.py --backbone dinov2_vitb14_reg --feature-tag pca256 --tsinghua-images /content/data/raw/tsinghua_dogs/low-resolution.zip --stanford-images /content/data/raw/stanford_dogs/Images --device cpu' > /content/work/logs/cross_overlap.log 2>&1 < /dev/null &

## 4. The audit's numbers and the provenance subsets

The provenance subsets of the Tsinghua test split come from `results/audit/cross_pairs.csv` alone, before any
test prediction exists (the rule is `provenance_subsets` in `scripts/subset_eval.py`):
* **stanford_origin**: a confirmed Stanford Dogs copy;
* **independent**: no candidate pair at all;
* **ambiguous**: a candidate but no confirmed pair (counted, not analysed).

The run recorded in `research_log/research_log.md` gave 3,115 / 1,362 / 723 of the 5,200 test images. From
here on, keep `results/audit/` as it is: it defines the subsets.

In [ ]:
import importlib.util, json
import pandas as pd
R, M = os.environ["DOGBREED_RESULTS"], os.environ["DOGBREED_MANIFESTS"]
r = json.load(open(f"{R}/audit/cross_tsinghua_stanford.json"))
print({k: r[k] for k in ("candidate_pairs", "confirmed_pairs", "confirmed_tsinghua_images",
                         "confirmed_stanford_images", "stanford_test_in_tsinghua_trainval", "stanford_test_clean")})
spec = importlib.util.spec_from_file_location("subset_eval", "scripts/subset_eval.py")
se = importlib.util.module_from_spec(spec)
spec.loader.exec_module(se)
m = pd.read_csv(f"{M}/tsinghua.csv", usecols=["id", "split"])
subs = se.provenance_subsets(pd.read_csv(f"{R}/audit/cross_pairs.csv", usecols=["t_id", "confirmed"]),
                             m.loc[m["split"] == "test", "id"])
print("Tsinghua test split by provenance:", {k: len(v) for k, v in subs.items()})
push()

## 5. Other backbones

The Tsinghua caches of ResNet-50, ConvNeXt-T and CLIP ViT-B/16 (orig and hflip only: they enter the comparison,
not the robustness study) are extracted by queue B in notebook 2 (jobs `extract_resnet50`,
`extract_convnext_tiny`, `extract_clip_vitb16`); queue A's backbone runs wait for the last of them.

## After a recycled runtime: rebuild the caches

The setup cell brings `/content/work` back from Drive, and the token PCAs with it; the feature caches are gone.
This cell rebuilds every cache the later steps read. It reuses the archives on Drive and the restored manifests
(`colab_data.sh all --no-prepare`) and the restored PCAs, and it does **not** re-run the overlap audit, whose
`cross_pairs.csv` defines the provenance subsets. Queue B does not redo extraction jobs it has marked done, so
the other backbones are rebuilt here too. The log ends with the last `EXTRACT DONE` line.

In [ ]:
R = os.environ["DOGBREED_RESULTS"]
assert os.path.exists(f"{R}/audit/cross_pairs.csv"), "no overlap audit yet: run sections 1-3 instead"
T, S = "/content/data/raw/tsinghua_dogs/low-resolution.zip", "/content/data/raw/stanford_dogs/Images"
steps = ["bash scripts/colab_data.sh all --no-prepare",
         f"bash scripts/extract_all.sh tsinghua dinov2_vitb14_reg {T} 256 224",
         f"ANN_VIEWS=bg_gray,bg_blur,crop_body bash scripts/extract_all.sh stanford dinov2_vitb14_reg {S} 256 224",
         f"python -I scripts/extract_features.py --dataset stanford --backbone dinov2_vitb14_reg --images {S} "
         "--splits test --views orig --token-pca 256 --pca-from tsinghua --workers 2"]
steps += [f"AUTO_VIEWS= ANN_VIEWS= bash scripts/extract_all.sh tsinghua {b} {T} 256 224"
          for b in ("resnet50", "convnext_tiny", "clip_vitb16")]
chain = " && ".join(steps)
!setsid nohup bash -c '{chain}' > /content/work/logs/rebuild_caches.log 2>&1 < /dev/null &

In [ ]:
!tail -n 3 /content/work/logs/rebuild_caches.log